In [3]:
import pandas as pd
import numpy as np
import os

base_dir = os.path.abspath(os.path.join(os.getcwd(), '../../'))
out_dir = os.path.join(base_dir, "data_out")
zip_access_file = os.path.join(out_dir, "Zip Access by Specialty Matrix (nearest office).csv")
pop_coverage_file = os.path.join(out_dir, "Population Coverage Index.csv")
sites_by_vol_file = os.path.join(out_dir, "Top Offices by Visit Volume.csv")
uc_drives_file = os.path.join(out_dir, "Urgent Care to Office Drive Times by Specialty (weighted average).csv")

top_volume = pd.read_csv(sites_by_vol_file)
pop_coverage = pd.read_csv(pop_coverage_file)
zip_matrix = pd.read_csv(zip_access_file)
uc_matrix = pd.read_csv(uc_drives_file, index_col = 0)

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\ajgho\\OneDrive\\Applications\\portfolio\\git hub\\upmc_senior_design_code\\senior-design-upmc-main\\data_out\\Top Offices by Visit Volume.csv'

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

hosp_vol = top_volume[top_volume['Location Type'] == 'Hospital'].copy()
com_vol = top_volume[top_volume['Location Type'] == 'Community'].copy()

top_n = 10

hosp_top = hosp_vol.head(top_n)
com_top = com_vol.head(top_n)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.barplot(data=hosp_top, x='Total Visits', y='Normalized Address', ax=axes[0], color='steelblue')
axes[0].set_title(f'Top {top_n} Hospital-Based Offices by Visit Volume')
axes[0].set_xlabel('Total Visits')
axes[0].set_ylabel('')

sns.barplot(data=com_top, x='Total Visits', y='Normalized Address', ax=axes[1], color='coral')
axes[1].set_title(f'Top {top_n} Community Offices by Visit Volume')
axes[1].set_xlabel('Total Visits')
axes[1].set_ylabel('')

max_visits = max(hosp_top['Total Visits'].max(), com_top['Total Visits'].max())

axes[0].set_xlim(0, max_visits * 1.05)
axes[1].set_xlim(0, max_visits * 1.05)

plt.tight_layout()
plt.show()

In [ ]:
styled = pop_coverage.style \
    .background_gradient(subset=['Good Access % (<15 min)'], cmap='Greens') \
    .background_gradient(subset=['Moderate Access % (15-30 min)'], cmap='YlOrBr') \
    .background_gradient(subset=['Limited Access % (30-45 min)'], cmap='Oranges') \
    .background_gradient(subset=['Access Desert % (>45 min)'], cmap='Reds') \
    .format({
        'Good Access % (<15 min)': '{:.1f}%',
        'Moderate Access % (15-30 min)': '{:.1f}%',
        'Limited Access % (30-45 min)': '{:.1f}%',
        'Access Desert % (>45 min)': '{:.1f}%'
    }) \
    .set_properties(**{'text-align': 'left'}) \
    .hide(axis='index')

styled

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
import sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../../')))  # project root for utils
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../../src/referral_ecosystem')))  # for three_ecosystems

from three_ecosystems import filter_routing_by_zips, NORTH_ZIPS, SOUTHWEST_ZIPS, EAST_ZIPS
import utils


upmc_df = pd.read_csv(os.path.join(out_dir, "UPMC Hospital, ASC, Urgent Care Locations.csv"))
zip_population = pd.read_csv(os.path.join(out_dir, "Census Data.csv"))
uc_routing = pd.read_csv(os.path.join(out_dir, "UC Routing Table.csv"))
pc_routing = pd.read_csv(os.path.join(out_dir, "PC to Specialty Routing Table.csv"))
patients_df = pd.read_csv(os.path.join(out_dir, "Office Data with Coordinates.csv"))
zip_growth_df = pd.read_csv(os.path.join(out_dir, "Zip Population Growth.csv"))


urgent_cares = upmc_df[upmc_df['Facility'] == 'Urgent Care'].copy()
urgent_cares["Zip"] = urgent_cares["Address"].str.extract(r'(\d{5})(?:\.0)?$')[0].astype(float).astype("Int64").astype(str).str.zfill(5)
target_zips = set(patients_df['Dept Zip'].astype(str).str.zfill(5))
urgent_cares_filtered = urgent_cares[urgent_cares["Zip"].isin(target_zips)]



region_name = 'Cranberry-Butler'
region_zips = NORTH_ZIPS

uc_region = filter_routing_by_zips(uc_routing, patients_df, urgent_cares_filtered, region_zips, 'UC Address')
pc_region = filter_routing_by_zips(pc_routing, patients_df, urgent_cares_filtered, region_zips, 'PC Address')

# Zip lookups
zip_lookup_pc = patients_df[['Normalized Address', 'Dept Zip']].drop_duplicates()
zip_lookup_pc['Dept Zip'] = zip_lookup_pc['Dept Zip'].astype(str).str.zfill(5)
pc_zip = dict(zip(zip_lookup_pc['Normalized Address'], zip_lookup_pc['Dept Zip']))
uc_zip = dict(zip(urgent_cares_filtered['Address'], urgent_cares_filtered['Zip'].astype(str).str.zfill(5)))
pop_lookup = zip_population.set_index('Zip Code')['Population'].to_dict()
pop_lookup = {str(k).zfill(5): v for k, v in pop_lookup.items()}
growth_lookup = {str(k).zfill(5): v for k, v in zip_growth_df.set_index('Zip Code')['Population Change Pct'].to_dict().items()}

# Build rows
rows = []
for addr in uc_region['UC Address'].unique():
    zip_code = uc_zip.get(addr, '')
    pop = pop_lookup.get(zip_code, 0)
    spec_times = uc_region[uc_region['UC Address'] == addr].set_index('Specialty')['Primary Drive Time (min)']
    row = {
        'Origin': f"[UC] {addr} | pop: {pop:,} | chg: {growth_lookup.get(zip_code, 0):+.1f}%",
        'Type': 'UC',
        'Population': pop
    }
    row.update(spec_times.to_dict())
    rows.append(row)

for addr in pc_region['PC Address'].unique():
    zip_code = pc_zip.get(addr, '')
    pop = pop_lookup.get(zip_code, 0)
    spec_times = pc_region[pc_region['PC Address'] == addr].set_index('Specialty')['Primary Drive Time (min)']
    row = {
        'Origin': f"[PC] {addr} | pop: {pop:,} | chg: {growth_lookup.get(zip_code, 0):+.1f}%",
        'Type': 'PC',
        'Population': pop
    }
    row.update(spec_times.to_dict())
    rows.append(row)

df = pd.DataFrame(rows).sort_values('Population', ascending=False).set_index('Origin')
specialty_cols = [c for c in df.columns if c not in ['Type', 'Population']]
heatmap_data = df[specialty_cols].astype(float)

# Plot
fig, ax = plt.subplots(figsize=(max(12, len(specialty_cols) * 0.8), max(8, len(df) * 0.45)))
sns.heatmap(
    heatmap_data,
    cmap='RdYlGn_r',
    annot=True,
    fmt='.0f',
    linewidths=0.5,
    cbar_kws={'label': 'Drive Time to Primary Clinic (min)'},
    ax=ax
)
ax.set_title(f'{region_name} — PC & UC to Specialty Drive Times', fontsize=13)
ax.set_xlabel('Specialty')
ax.set_ylabel('')
ax.tick_params(axis='x', rotation=45)
ax.tick_params(axis='y', rotation=0, labelsize=7)

plt.tight_layout()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\ajgho\\OneDrive\\Applications\\portfolio\\git hub\\upmc_senior_design_code\\senior-design-upmc-main\\data_out\\UPMC Hospital, ASC, Urgent Care Locations.csv'

In [ ]:
region_name = 'Southwest'
region_zips = SOUTHWEST_ZIPS

uc_region = filter_routing_by_zips(uc_routing, patients_df, urgent_cares_filtered, region_zips, 'UC Address')
pc_region = filter_routing_by_zips(pc_routing, patients_df, urgent_cares_filtered, region_zips, 'PC Address')

# Zip lookups
zip_lookup_pc = patients_df[['Normalized Address', 'Dept Zip']].drop_duplicates()
zip_lookup_pc['Dept Zip'] = zip_lookup_pc['Dept Zip'].astype(str).str.zfill(5)
pc_zip = dict(zip(zip_lookup_pc['Normalized Address'], zip_lookup_pc['Dept Zip']))
uc_zip = dict(zip(urgent_cares_filtered['Address'], urgent_cares_filtered['Zip'].astype(str).str.zfill(5)))
pop_lookup = zip_population.set_index('Zip Code')['Population'].to_dict()
pop_lookup = {str(k).zfill(5): v for k, v in pop_lookup.items()}
growth_lookup = {str(k).zfill(5): v for k, v in zip_growth_df.set_index('Zip Code')['Population Change Pct'].to_dict().items()}

# Build rows
rows = []
for addr in uc_region['UC Address'].unique():
    zip_code = uc_zip.get(addr, '')
    pop = pop_lookup.get(zip_code, 0)
    spec_times = uc_region[uc_region['UC Address'] == addr].set_index('Specialty')['Primary Drive Time (min)']
    row = {
        'Origin': f"[UC] {addr} | pop: {pop:,} | chg: {growth_lookup.get(zip_code, 0):+.1f}%",
        'Type': 'UC',
        'Population': pop
    }
    row.update(spec_times.to_dict())
    rows.append(row)

for addr in pc_region['PC Address'].unique():
    zip_code = pc_zip.get(addr, '')
    pop = pop_lookup.get(zip_code, 0)
    spec_times = pc_region[pc_region['PC Address'] == addr].set_index('Specialty')['Primary Drive Time (min)']
    row = {
        'Origin': f"[PC] {addr} | pop: {pop:,} | chg: {growth_lookup.get(zip_code, 0):+.1f}%",
        'Type': 'PC',
        'Population': pop
    }
    row.update(spec_times.to_dict())
    rows.append(row)

df = pd.DataFrame(rows).sort_values('Population', ascending=False).set_index('Origin')
specialty_cols = [c for c in df.columns if c not in ['Type', 'Population']]
heatmap_data = df[specialty_cols].astype(float)

# Plot
fig, ax = plt.subplots(figsize=(max(12, len(specialty_cols) * 0.8), max(8, len(df) * 0.45)))
sns.heatmap(
    heatmap_data,
    cmap='RdYlGn_r',
    annot=True,
    fmt='.0f',
    linewidths=0.5,
    cbar_kws={'label': 'Drive Time to Primary Clinic (min)'},
    ax=ax
)
ax.set_title(f'{region_name} — PC & UC to Specialty Drive Times', fontsize=13)
ax.set_xlabel('Specialty')
ax.set_ylabel('')
ax.tick_params(axis='x', rotation=45)
ax.tick_params(axis='y', rotation=0, labelsize=7)


plt.tight_layout()
plt.show()

In [ ]:
region_name = 'East'
region_zips = EAST_ZIPS

uc_region = filter_routing_by_zips(uc_routing, patients_df, urgent_cares_filtered, region_zips, 'UC Address')
pc_region = filter_routing_by_zips(pc_routing, patients_df, urgent_cares_filtered, region_zips, 'PC Address')

# Zip lookups
zip_lookup_pc = patients_df[['Normalized Address', 'Dept Zip']].drop_duplicates()
zip_lookup_pc['Dept Zip'] = zip_lookup_pc['Dept Zip'].astype(str).str.zfill(5)
pc_zip = dict(zip(zip_lookup_pc['Normalized Address'], zip_lookup_pc['Dept Zip']))
uc_zip = dict(zip(urgent_cares_filtered['Address'], urgent_cares_filtered['Zip'].astype(str).str.zfill(5)))
pop_lookup = zip_population.set_index('Zip Code')['Population'].to_dict()
pop_lookup = {str(k).zfill(5): v for k, v in pop_lookup.items()}
growth_lookup = {str(k).zfill(5): v for k, v in zip_growth_df.set_index('Zip Code')['Population Change Pct'].to_dict().items()}

# Build rows
rows = []
for addr in uc_region['UC Address'].unique():
    zip_code = uc_zip.get(addr, '')
    pop = pop_lookup.get(zip_code, 0)
    spec_times = uc_region[uc_region['UC Address'] == addr].set_index('Specialty')['Primary Drive Time (min)']
    row = {
        'Origin': f"[UC] {addr} | pop: {pop:,} | chg: {growth_lookup.get(zip_code, 0):+.1f}%",
        'Type': 'UC',
        'Population': pop
    }
    row.update(spec_times.to_dict())
    rows.append(row)

for addr in pc_region['PC Address'].unique():
    zip_code = pc_zip.get(addr, '')
    pop = pop_lookup.get(zip_code, 0)
    spec_times = pc_region[pc_region['PC Address'] == addr].set_index('Specialty')['Primary Drive Time (min)']
    row = {
        'Origin': f"[PC] {addr} | pop: {pop:,} | chg: {growth_lookup.get(zip_code, 0):+.1f}%",
        'Type': 'PC',
        'Population': pop
    }
    row.update(spec_times.to_dict())
    rows.append(row)

df = pd.DataFrame(rows).sort_values('Population', ascending=False).set_index('Origin')
specialty_cols = [c for c in df.columns if c not in ['Type', 'Population']]
heatmap_data = df[specialty_cols].astype(float)

# Plot
fig, ax = plt.subplots(figsize=(max(12, len(specialty_cols) * 0.8), max(8, len(df) * 0.45)))
sns.heatmap(
    heatmap_data,
    cmap='RdYlGn_r',
    annot=True,
    fmt='.0f',
    linewidths=0.5,
    cbar_kws={'label': 'Drive Time to Primary Clinic (min)'},
    ax=ax
)
ax.set_title(f'{region_name} — PC & UC to Specialty Drive Times', fontsize=13)
ax.set_xlabel('Specialty')
ax.set_ylabel('')
ax.tick_params(axis='x', rotation=45)
ax.tick_params(axis='y', rotation=0, labelsize=7)

plt.tight_layout()
plt.show()

In [ ]:
import re

pop_lookup = {str(k).zfill(5): v for k, v in zip_population.set_index('Zip Code')['Population'].to_dict().items()}

def extract_zip(addr):
    match = re.search(r'(\d{5})\s*(?:,\s*USA)?$', addr.strip())
    return match.group(1) if match else ''

# Sort first on original index, then rename
uc_matrix['_pop'] = [pop_lookup.get(extract_zip(addr), 0) for addr in uc_matrix.index]
uc_matrix = uc_matrix.sort_values('_pop', ascending=False).drop(columns='_pop')

uc_matrix.index = [
    f"{addr} | pop: {pop_lookup.get(extract_zip(addr), 0):,}"
    for addr in uc_matrix.index
]

plt.figure(figsize=(16, 12))
sns.heatmap(
    uc_matrix,
    cmap='RdYlGn_r',
    annot=True,
    fmt='.0f',
    linewidths=0.5,
    cbar_kws={'label': 'Weighted Avg Drive Time (min)'}
)
plt.title('Urgent Care to Specialty Drive Times (minutes)')
plt.xlabel('Specialty')
plt.ylabel('Urgent Care Location')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()